<a href="https://colab.research.google.com/github/Copperhorse/Profanity_Distilbert_LORA/blob/main/creating_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install unsloth_zoo

  Using cached nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
Using cached nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_x86_64.whl (594.3 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xformers 0.0.29.post3 requires torch==2.6.0, but you have torch 2.9.1 which is incompatible.
torchaudio 2.9.0+cu126 requires torch==2.9.0, but you have torch 2.9.1 which is incompatible.
torchvision 0.24.0+cu126 requires torch==2.9.0, but you have torch 2.9.1 which is incompatible.


In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth==2025.8.9

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel
import torch

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Load the base model and tokenizer
model_path = '/content/drive/MyDrive/profanity_classifier_distilbert_lora'
base_model_name = 'distilbert-base-uncased'  # or whatever base you used

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_name,
    num_labels=2
)

# Load the LoRA adapter
model = PeftModel.from_pretrained(base_model, model_path)
model.eval()  # Set to evaluation mode

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

Mounted at /content/drive


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def predict_profanity(text, threshold=0.9):
    """
    Predict if text contains profanity with confidence threshold

    Args:
        text: Input text to classify
        threshold: Minimum probability to classify as profanity (default 0.9 for high precision)

    Returns:
        dict with prediction, probability, and whether to include in training
    """
    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=512,
        padding=True
    ).to(device)

    # Get prediction
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        profanity_prob = probs[0][1].item()  # Probability of class 1 (profanity)

    # Only include high-confidence predictions
    has_profanity = profanity_prob > threshold
    include_in_training = profanity_prob > threshold or profanity_prob < (1 - threshold)

    return {
        'has_profanity': has_profanity,
        'profanity_probability': profanity_prob,
        'include_in_training': include_in_training
    }

In [ ]:
import pandas as pd
from tqdm import tqdm

# Load your reviews dataset
# This could be Amazon reviews, Yelp, etc.
df = pd.read_csv('/content/Amazon.csv')  # Adjust path and format

# Apply profanity classifier to each review
results = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    review_text = row['text']  # Adjust column name

    # Get profanity prediction
    pred = predict_profanity(review_text, threshold=0.9)

    # Combine with original data
    result = {
        'text': review_text,
        'has_profanity': pred['has_profanity'],
        'profanity_probability': pred['profanity_probability'],
        'sentiment': row.get('sentiment', None),  # If you have it
        'rating': row.get('label', None),  # If you have it
        'include_in_training': pred['include_in_training']
    }

    results.append(result)

# Create new dataframe with predictions
df_labeled = pd.DataFrame(results)

# Filter to only high-confidence predictions for training
df_training = df_labeled[df_labeled['include_in_training'] == True]

print(f"Total reviews: {len(df_labeled)}")
print(f"Reviews with profanity: {df_labeled['has_profanity'].sum()}")
print(f"Reviews for training: {len(df_training)}")

100%|██████████| 210000/210000 [26:38<00:00, 131.34it/s]


Total reviews: 210000
Reviews with profanity: 682
Reviews for training: 202889


In [ ]:
# Separate reviews with and without profanity
df_profanity = df_labeled[df_labeled['has_profanity'] == True]
df_no_profanity = df_labeled[df_labeled['has_profanity'] == False]

# Determine the number of profanity reviews
num_profanity = len(df_profanity)

# Sample an equal number of non-profanity reviews
# Ensure we don't try to sample more than available
num_to_sample = min(num_profanity, len(df_no_profanity))
df_no_profanity_sampled = df_no_profanity.sample(n=num_to_sample, random_state=42)

# Concatenate to create a balanced dataset
df_balanced = pd.concat([df_profanity, df_no_profanity_sampled])

# Shuffle the new DataFrame to mix profanity and non-profanity reviews
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total reviews in balanced dataset: {len(df_balanced)}")
print(f"Reviews with profanity in balanced dataset: {df_balanced['has_profanity'].sum()}")
print(f"Reviews without profanity in balanced dataset: {len(df_balanced) - df_balanced['has_profanity'].sum()}")

df_balanced.head()

Total reviews in balanced dataset: 1364
Reviews with profanity in balanced dataset: 682
Reviews without profanity in balanced dataset: 682


,text,has_profanity,profanity_probability,sentiment,rating,include_in_training
0,crawls up the butt. hard to sleep in\n\nCute b...,True,0.971946,negative,1,True
1,Tip works the filter not so much\n\nThe Tip wo...,False,0.022020,positive,3,True
2,Cute\n\nThe little owl timer is so cute I almo...,False,0.006371,positive,4,True
3,Idiotic assembly required.\n\nI buy a lot of t...,True,0.939297,negative,1,True
4,🤣\n\nHell naw skirt was bogus ass hell a joke ...,True,0.986663,negative,1,True


In [ ]:
df_training = df_balanced

In [ ]:
def create_llm_training_sample(row):
    """
    Create instruction-format training data for LLM
    """
    review = row['text']  # Fixed: was 'review_text'
    has_profanity = row['has_profanity']
    sentiment = row.get('sentiment', 'neutral')

    instruction = f"""Analyze the following customer review and provide:
1. Whether it contains profanity (yes/no)
2. The sentiment (positive/negative/neutral)
3. If profanity is present, rewrite it in a polite, professional manner while preserving the meaning and sentiment.

Review: {review}"""

    # Create output WITHOUT placeholder
    if has_profanity:
        output = f"""Profanity: yes
Sentiment: {sentiment}
Rewritten: """  # LLM will learn to generate this
    else:
        output = f"""Profanity: no
Sentiment: {sentiment}
Rewritten: Not needed"""

    return {
        'instruction': instruction,
        'input': '',
        'output': output
    }

# Generate training samples
training_samples = df_training.apply(create_llm_training_sample, axis=1).tolist()

# Convert to DataFrame
df_llm_training = pd.DataFrame(training_samples)

# Save
df_llm_training.to_json('synthetic_training_data.jsonl', orient='records', lines=True)

In [ ]:
# Inspect your data
print(df_llm_training.head())
print(f"\nDataset shape: {df_llm_training.shape}")
print(f"Columns: {df_llm_training.columns.tolist()}")

# Check balance
print(f"\nProfanity distribution:")
print(df_training['has_profanity'].value_counts())

                                         instruction input  \
0  Analyze the following customer review and prov...         
1  Analyze the following customer review and prov...         
2  Analyze the following customer review and prov...         
3  Analyze the following customer review and prov...         
4  Analyze the following customer review and prov...         

                                              output  
0   Profanity: yes\nSentiment: negative\nRewritten:   
1  Profanity: no\nSentiment: positive\nRewritten:...  
2  Profanity: no\nSentiment: positive\nRewritten:...  
3   Profanity: yes\nSentiment: negative\nRewritten:   
4   Profanity: yes\nSentiment: negative\nRewritten:   

Dataset shape: (1364, 3)
Columns: ['instruction', 'input', 'output']

Profanity distribution:
has_profanity
True     682
False    682
Name: count, dtype: int64


In [ ]:
from datasets import Dataset

# Convert to HuggingFace dataset format
def format_prompt(sample):
    """Format for instruction fine-tuning"""
    return {
        'text': f"""### Instruction:
{sample['instruction']}

### Response:
{sample['output']}"""
    }

# Apply formatting
formatted_data = [format_prompt(sample) for sample in training_samples]
dataset = Dataset.from_list(formatted_data)

# Split into train/validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f"Training samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['test'])}")

# Save
dataset.save_to_disk('/content/drive/MyDrive/llm_training_dataset')

Training samples: 1227
Validation samples: 137


Saving the dataset (0/1 shards):   0%|          | 0/1227 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/137 [00:00<?, ? examples/s]

In [ ]:


# Load model in 4-bit
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="LiquidAI/LFM2.5-1.2B-Instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,  # Enable 4-bit quantization
)

# Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

training_args = SFTConfig(
    output_dir="./lfm2-unsloth-qlora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    bf16=True,
)

dataset = load_dataset("your-dataset")

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    tokenizer=tokenizer,
)

trainer.train()